# Prediction Plots: GT, MLP, And Cov Fit

This notebook shows the plotting workflow for generated patho SNR signals. It either runs the MLP comparison from external checkpoints or reloads an existing `patho_comparison.csv` and redraws the scenario figures.

The orange covariance-fit series appears only when `COV_FIT_ROOT` points to folders containing `SNR30/signal_real_XXX/wlls/dtd_covariance_dps.mat`.

## 1. Local Setup

Run this first from the repo checkout. It makes `src/` importable without installing the package.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "qti_unified").exists():
    REPO_ROOT = Path(r"C:/QTI_Unified")

SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Repo:", REPO_ROOT)

## 2. Point To Data, Models, And Cov Fits

`DATA_ROOT` must contain `Results_2_MLP_patho` and `Results_SNR_fit_2_MLP_patho`. Use `C:/SynQTI-IR/data` for the old generated batch, or `C:/QTI_Unified/data` if you generated the batch with this repo.

In [ ]:
DATA_ROOT = Path(r"C:/SynQTI-IR/data")
# DATA_ROOT = REPO_ROOT / "data"

MODEL_ROOT = Path(r"C:/QTI_ML/BENCHMARK_ABSTRACT")
COV_FIT_ROOT = Path(r"C:/SynQTI-IR/data/Fit_Results/dtd_covariance_snr30_batch_sub_min_pp_patho")
SNR_FOLDER = "SNR30"
DEVICE = "cpu"

OUTPUT_ROOT = DATA_ROOT / "runs" / "QTI_MLP_synthetic_compare_patho"

for label, path in {
    "DATA_ROOT": DATA_ROOT,
    "GT root": DATA_ROOT / "Results_2_MLP_patho",
    "SNR root": DATA_ROOT / "Results_SNR_fit_2_MLP_patho",
    "MODEL_ROOT": MODEL_ROOT,
    "COV_FIT_ROOT": COV_FIT_ROOT,
    "OUTPUT_ROOT": OUTPUT_ROOT,
}.items():
    print(f"{label:12s}: {path} | exists={path.exists()}")

## 3. Option A: Run MLP Prediction And Plot

Set `RUN_MLP = True` when the benchmark checkpoints, Torch, pandas, and generated SNR files are available. This writes `patho_comparison.csv` and the scenario PNGs into `OUTPUT_ROOT`.

In [ ]:
RUN_MLP = True

if RUN_MLP:
    from qti_unified.pipeline import compare_patho_predictions

    df = compare_patho_predictions(
        data_root=DATA_ROOT,
        model_root=MODEL_ROOT,
        cov_fit_root=COV_FIT_ROOT,
        snr_folder=SNR_FOLDER,
        output_root=OUTPUT_ROOT,
        device=DEVICE,
    )
    print("Rows:", len(df))
    display(df.head())
else:
    print("Skipped. Set RUN_MLP = True to run checkpoints and create a fresh comparison CSV/plots.")

## 4. Option B: Reload Existing CSV And Replot

Use this when `patho_comparison.csv` already exists. This does not need to rerun the MLP.

In [ ]:
plot_paths = []
comparison_csv = OUTPUT_ROOT / "patho_comparison.csv"

if comparison_csv.exists():
    import pandas as pd
    from qti_unified.plotting import INVAR_KEYS, plot_scenario_comparisons

    df = pd.read_csv(comparison_csv)
    plot_paths = plot_scenario_comparisons(
        df,
        out_dir=OUTPUT_ROOT / "plots_from_csv",
        invar_keys=INVAR_KEYS,
    )
    print("Rows:", len(df))
    print("Written plots:")
    for path in plot_paths:
        print("-", path)
    display(df.head())
else:
    print("No comparison CSV found:", comparison_csv)
    print("Run Option A first, or set DATA_ROOT/OUTPUT_ROOT to a folder that already has patho_comparison.csv.")

## 5. Show Plots In The Notebook

In [ ]:
from pathlib import Path
from IPython.display import Image, Markdown, display

OPEN_IN_SYSTEM_VIEWER = False  # Set True on Windows to also open PNGs in image-viewer windows.

known_plot_paths = [Path(p) for p in globals().get("plot_paths", [])]
for folder in [OUTPUT_ROOT, OUTPUT_ROOT / "plots_from_csv", OUTPUT_ROOT / "manual_plots"]:
    if folder.exists():
        known_plot_paths.extend(sorted(folder.glob("*_patho_cov.png")))

plot_paths = []
seen = set()
for path in known_plot_paths:
    path = Path(path)
    if path.exists() and path not in seen:
        plot_paths.append(path)
        seen.add(path)

if plot_paths:
    print(f"Showing {len(plot_paths)} plot(s).")
    for path in plot_paths:
        title = path.stem.replace("_patho_cov", "").replace("_", " ").title()
        display(Markdown(f"### {title}"))
        display(Image(filename=str(path)))
        if OPEN_IN_SYSTEM_VIEWER:
            import os

            os.startfile(path)
else:
    print("No plot PNGs found yet. Run Option A or Option B first, then rerun this cell.")

## 6. Under The Hood: Discovery, Prediction, Table, Plot

This is the same workflow expanded into separate calls. It is useful for checking whether cov-fit paths are found before running the MLP.

In [ ]:
from qti_unified.pipeline import discover_prediction_cases

rows = discover_prediction_cases(
    data_root=DATA_ROOT,
    cov_fit_root=COV_FIT_ROOT,
    snr_folder=SNR_FOLDER,
)

print("Prediction rows:", len(rows))
print("Rows with cov fit:", sum(bool(row.get("cov_dps_exists")) for row in rows))
if rows:
    sample = rows[0].copy()
    print("Example signal:", sample["signal_path"])
    print("Example GT:", sample["gt_json"])
    print("Example cov fit:", sample["cov_dps_path"])
    print("Cov exists:", sample["cov_dps_exists"])

In [ ]:
RUN_MANUAL_MLP = False

if RUN_MANUAL_MLP:
    from qti_unified.mlp import INVAR_KEYS, collect_benchmark_model_paths, ensemble_predict
    from qti_unified.plotting import build_comparison_table, plot_scenario_comparisons

    model_paths = collect_benchmark_model_paths(MODEL_ROOT)
    signal_paths = [row["signal_path"] for row in rows]
    predictions = ensemble_predict(signal_paths, model_paths, invar_keys=INVAR_KEYS, device=DEVICE)
    df_manual = build_comparison_table(rows, predictions, invar_keys=INVAR_KEYS)
    manual_plot_paths = plot_scenario_comparisons(df_manual, OUTPUT_ROOT / "manual_plots", invar_keys=INVAR_KEYS)
    display(df_manual.head())
    print("Manual plot count:", len(manual_plot_paths))
else:
    print("Skipped. This cell shows the exact lower-level calls used by compare_patho_predictions.")

## Figure Semantics

- Black dashed line or star: stored ground truth from `*_GT_params.json`.
- Blue circles: MLP mean across SNR realizations, with standard deviation error bars.
- Orange triangles: covariance-fit mean across SNR realizations, with standard deviation error bars. Missing cov-fit files are ignored as NaN.
- Each scenario gets one PNG containing `MD`, `FA`, `uFA`, `C_c`, and `C_MD` panels.